# Toy Model for NeurIPS 2026 Submission: 
# Diffusion Models Memorize in Training -- and Generalize in Inference
$\newcommand{\e}{\boldsymbol{\varepsilon}}\newcommand{\x}{\boldsymbol{x}}\newcommand{\y}{\boldsymbol{y}}$

We study a 2D toy diffusion model to isolate the mechanism behind the noise-dependent relative generalization gap observed in real diffusion models under an L2 denoising metric. The goal is not to reproduce image statistics, but to capture the interaction between posterior geometry and noise-dependent dynamics in a controlled, analytically accessible setting.

As data distribution, we choose a circle in $\mathbb{R}^2$. This choice has two properties relevant to our analysis:
1. It has no privileged direction, mimicking the isotropy of high-dimensional data distributions.
2. The superposition of samples lies far outside the data manifold, avoiding denoising interactions that are not possible for real diffusion models.

Training and validation points are sampled i.i.d. from this circle and split evenly.

---

## Posterior mean predictor

Let $\{y_i\}_{i=1}^N$ denote $N$ training samples. We can generate trajectories $\x(t)$ by numerically solving the ordinary differential equation $d\x(t) = \e^*(\x(t), \sigma(t))d\sigma(t)$, with $\e^*$ the optimal noise predictor in the Bayesian sense, and $\sigma(t)\in [0,\sigma_{max}]$ the current noise level. Note that $d\sigma(t)<0$ along a denoising trajectory. [It can be shown](https://arxiv.org/abs/2206.00364) that for a finite set of $N$ data points, the optimal noise predictor takes the explicit form
\begin{equation}
    \e^*(\x, \sigma(t)) = \frac{1}{\sigma(t)} \sum_{i=1}^N (\x-\y_i)p(\y_i|\x,t), \quad p(\y_i|\x,t) = \frac{{\cal N}(\x|\y_i,\sigma(t)^2)}{\sum_{i=1}^N{\cal N}(\x|\y_i,\sigma(t)^2)},
\end{equation}
where ${\cal N}(.|.)$ denotes an isotropic normal distribution. That is, the optimal noise predictor is the normalised residual between $\x$ and the posterior-mean $\sum_{i=1}^N \y_i p(\y_i|\x,t)$ as an estimator for a data point. The optimal noise predictor generates trajectories with endpoints arbitrarily close to one of the data points if the ODE is initialized with $\x_{init}\sim {\cal N}(\x|\textbf{0},\sigma_{max}^2)$ and $\sigma_{max}$ is sufficiently large. 

This defines a smooth vector field over $\mathbb{R}^2$ whose geometry depends on $\tilde{\sigma}$: for small $\tilde{\sigma}$, the predictor is highly localized around individual training points; for large $\tilde{\sigma}$, it becomes global and symmetric.

---

## Modeling an imperfect diffusion model

To construct an error-prone denoiser, we substitute the true data distribution with a “broader” data distribution $P(y) → P_\delta(y)$, with $_\delta(y) = \int_{\mathbb{R}^d} \mathcal{N}(y|y′,δ^2)P(y')dy′$, where $\delta$ is the standard deviation of noise that randomly shifts the true data points. [The corresponding posterior mean predictor is given by](https://arxiv.org/abs/2411.10257)

$$
y_\delta(x,\sigma)
=
\frac{\delta^2 x + \sigma^2\, y^*(x,\tilde{\sigma})}
     {\sigma^2+\delta^2},
\qquad
\tilde{\sigma}^2 = \sigma^2 + \delta^2.
$$

The prediction error decomposes naturally into two factors:

$$
y_\delta(x,\sigma) - x
=
\frac{\sigma^2}{\sigma^2+\delta^2}
\bigl(y^*(x,\tilde{\sigma}) - x\bigr).
$$

This separates:

- **Geometry:**  
  $y^*(x,\tilde{\sigma}) - x$, determined by the arrangement of training points and the scale $\tilde{\sigma}$.

- **Dynamics:**  
  $\frac{\sigma^2}{\sigma^2+\delta^2}$, a scalar that controls how strongly this geometry is expressed.

<!-- Note two key properties of this predictor:

1. **Identity-like at now noise:**  
   If $\delta > 0$, as $\sigma \to 0$,
   $$
   y_\delta(x,\sigma) \to x,
   $$

2. **Frozen posterior geometry:**  
The posterior mean is evaluated at noise scale $\tilde{\sigma}$, so even as $\sigma \to 0$, $y_\delta$ never becomes a projection onto the training points. -->

---

## $\sigma$-$\delta$-dependent regimes

**($\sigma \gg \delta$)**  
$\y_\delta$ behaves like the posterior mean predictor $\y^*$, but noisy samples $\x$ are far away from the data manifold. Predictions tend towards the superposition of the training data, regardless of $\y$ being a training or validation sample, resulting in similar average errors between training and validation samples. 

**($\sigma \ll \delta$)**  
The prefactor $\frac{\sigma^2}{\sigma^2+\delta^2}$ dampens the dynamics, causing $y_\delta$ to behave identity-like ($\y_\delta(\x,\sigma) \approx \x$). In addition, the flow field localizes more slowly around training points, because $\y^*$ is evaluated at $\tilde{\sigma}$ and not at $\sigma$. 
Again, average errors are very similar. 

**($\sigma \approx \delta$)**  
Neither are dynamics suppressed nor are samples far off the manifold. Crucially, the error parameter $\delta$ determines where in the flow field the transition from ($\sigma \gg \delta$) to ($\sigma \ll \delta$) happens. For large $\delta$, the generalization gap never emerges because the predictor transitions from global averaging to identity-like behavior without ever entering highly localized regions of the flow field. For sufficiently small $\delta$, a generalization gap emerges at $\sigma \approx \delta$, because $\x(t)$ is close enough to the data manifold for the posterior mean to be attracted to the specific locations of training points, and the prefactor does not yet suppress the dynamics. 

---

# 1. Setup
<a id='setup'></a>
Ensure you have the following packages installed in your current environment and start exploring the notebook:
- Numpy
- Matplotlib
- tqdm

Using the `save_as` parameter in `model_and_metric` will automatically save a plot for each combination of $\sigma$ and $\delta$ and not plot the full grid here.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms

from toy_utils import *

plt.style.use('default')

%load_ext autoreload
%autoreload 2

In [ ]:
# Sampler settings 
num_steps, sigma_min, sigma_max, rho = 120, 2e-3, 80, 7  # Number of steps, min noise level, max noise level, sampler curvature

## Flow field, gap contours and displacement regions

Select a mode below to generate 2 different visualizations:

1. **Flow**: Shows the flow field lines and predictions of the optimal target predictor $\y^*(\x, \sigma)$ at different noise levels $\sigma = 28, 2.8, 0.63$. Color indicates the magnitude of the prediction error $\y^*(\x, \sigma) - x$.
2. **Displacement**: Shows contours of the displacement error $||\y_\delta(\x, \sigma) - x||$. These regions indicate where the predictor behaves similarly on training and validation samples and therefore contributes little to the L2 generalization gap.

Select any combinations of $\sigma$ and $\delta$ via the `sigma_idx` and `all_deltas` variables. The resulting plot will be a grid with $\delta$ fixed per row and $\sigma$ fixed per column.

In [ ]:
random_data = False
mode = 'flow'  # 'flow', 'displacement'

if mode == 'flow' or mode == 'displacement':
    N = 10 if not random_data else 20
    plot_points = N
    sigma_idx = [42, 60, 96]  # which sigmas to plot
    all_deltas = np.array([0e-2])
elif mode == 'error':
    N = 10
    plot_points = 0
    sigma_idx = [16]  # which sigmas to plot
    all_deltas = np.array([20e-2, 16e-2, 12e-2])
    
data_str, N, data, data_labels, data_split = prepare_data(random_data, 
                                                          N=N, 
                                                          split=False,  # use only training points
                                                          offset=1e-3,  # rotate circle slightly
                                                         )

scale = 10
data = data * scale
window_size = 1.5 * scale
all_deltas = all_deltas * scale

plt.figure(figsize=(4*len(sigma_idx), 4*len(all_deltas)))
plt.subplots_adjust(hspace=0, wspace=0)
plot_counter = 1
for i, delta in enumerate(all_deltas):
    print(f"Delta: {delta}")
    _, plot_counter, _ = model_and_metric(data, data_split, data_labels, num_steps, sigma_min, sigma_max, rho, delta, window_size, 
                                          samples_per_point=100, 
                                          sigma_idx=sigma_idx, 
                                          plot_counter=plot_counter, 
                                          plot=True, 
                                          contours=True, 
                                          threshold=False, 
                                          plot_points=plot_points, 
                                          plot_samples_per_point=100, 
                                          n_rows=len(all_deltas),
                                          plot_sigma_idx=sigma_idx,
                                          mode=mode, 
                                          scale=scale,
                                          )
    
plt.show()
plt.close()

In [ ]:
random_data = False
mode = 'error'

plot_points = 0
sigma_idx = [48]  # which sigmas to plot
all_deltas = np.array([20e-2, 16e-2, 12e-2])
N = 10
plt.figure(figsize=(5.5,5))
    
data_str, N, data, data_labels, data_split = prepare_data(random_data, N=N, split=False, offset=1e-3)

scale = 10
data = data * scale
window_size = 1.5 * scale
all_deltas = all_deltas * scale
dim = data.shape[1]

plt.subplots_adjust(hspace=0, wspace=0)
plot_counter = 1
for i, delta in enumerate(all_deltas):
    print(f"Delta: {delta}")
    _, plot_counter, t_steps = model_and_metric(data, data_split, data_labels, num_steps, sigma_min, sigma_max, rho, delta, window_size, 
                                                samples_per_point=100, 
                                                sigma_idx=sigma_idx, 
                                                plot_counter=plot_counter, 
                                                plot=True, 
                                                contours=True, 
                                                threshold=False, 
                                                plot_points=plot_points, 
                                                plot_samples_per_point=100, 
                                                n_rows=len(all_deltas),
                                                plot_sigma_idx=sigma_idx,
                                                mode=mode, 
                                                scale=scale,
                                                )
    
training_angles = np.arctan2(data[1:,1], data[1:,0]) + np.pi  # in range (0,2pi)
plt.vlines(training_angles, 0, 10, color='black', linestyle='--', label='Training points', alpha=0.5)
plt.legend()
plt.ylim([0, 1.3]) if not random_data else plt.ylim([0,6])
plt.show()
plt.close()

## Relative generalization gap vs model error $\delta$

Set a range for the model error parameter $\delta$ in `all_deltas` to generate a plot of the relative generalization gap across $\sigma$.

In [ ]:
random_data = False
data_str, N, data, data_labels, data_split = prepare_data(random_data)
all_deltas =  np.linspace(22e-2, 12e-2, 11, endpoint=True) # random: [1.8e-1, 1.3e-1, 7e-2]  symm: [1.2e-1, 1e-1, 8e-2]

scale = 10
data = data * scale
window_size = 2 * scale
all_deltas = all_deltas * scale

all_errors = np.zeros((len(all_deltas), num_steps))
for i, delta in enumerate(all_deltas):
    relative_error, _, t_steps = model_and_metric(data, data_split, data_labels, num_steps, sigma_min, sigma_max, rho, delta, window_size, 
                                                  samples_per_point=50, sigma_idx=np.arange(num_steps), plot=False)
    all_errors[i] = relative_error

# Make generalization gap plot
plt.figure(figsize=(6,5))
def fmt(x, pos):
    return f'{x:.1f}'  # Format tick labels

gen_gap_plot(t_steps, 
             all_errors, 
             color_range=all_deltas, 
             cbar_ticks=all_deltas[::2], 
             ylim=[-0.1, 1.0] if not random_data else [-0.1, 0.5], 
             fmt=fmt, 
            )

## Relative generalization gap vs guidance weights (Autoguidance)

Set a range for the guidance weights in `guid_weights` to generate a plot of the relative genralization gap across $\sigma$. We use Autoguidance, where the positive and negative model differ only in their model error parameter $\delta$, specified in `delta_pos` and `delta_neg`.

In [ ]:
random_data = False
data_str, N, data, data_labels, data_split = prepare_data(random_data)

scale = 10
data = data * scale
window_size = 2 * scale

delta_pos = 18e-2 * scale
delta_neg = 22e-2 * scale
guid_weights = np.arange(0, 2.2, 0.2).tolist()

all_errors = np.zeros((len(guid_weights), num_steps))
for i, g_weight in enumerate(guid_weights):
    relative_error, _, t_steps = model_and_metric(data, data_split, data_labels, num_steps, sigma_min, sigma_max, rho, delta_pos, window_size, 
                                                  guid_weight=g_weight, delta_neg=delta_neg,
                                                  samples_per_point=50, sigma_idx=np.arange(num_steps), plot=False)
    all_errors[i] = relative_error


# Make generalization gap plot
plt.figure(figsize=(6,5))
def fmt(x, pos):
    return f'{x:.1f}'  # Format tick labels

gen_gap_plot(t_steps, 
             all_errors, 
             color_range=guid_weights, 
             cbar_ticks=guid_weights, 
             ylim=[-0.1, 1.0] if not random_data else [-0.1, 0.5], 
             fmt=fmt, 
             cmap='viridis',
            )

## Relative generalization gap vs number of training samples

Set a range for the number of training and validation samples $N$ in `all_N` to generate a plot of the relative generalization gap across $\sigma$. Note, that the ticks on the colorbar correspond to the number of training samples $N/2$.

In [ ]:
random_data = False

all_N = np.linspace(16, 30, 8) if not random_data else np.linspace(20, 60, 5)

all_errors = np.zeros((len(all_N), num_steps))
all_deltas = [12e-2]
sigma_idx = np.arange(num_steps)

scale = 10
delta =  16e-2 * scale
window_size = 2 * scale

plt.figure(figsize=(4*len(sigma_idx), 4*len(all_deltas)))
plt.subplots_adjust(hspace=0, wspace=0)
plot_counter = 1

for i, N in enumerate(all_N):
    data_str, N, data, data_labels, data_split = prepare_data(random_data, N=int(N), seed=2)
    
    data = data * scale
    plt.figure(figsize=(4,4*len(all_N)))
    relative_error, _, t_steps = model_and_metric(data, data_split, data_labels, num_steps, sigma_min, sigma_max, rho, delta, window_size, 
                                                  samples_per_point=50, sigma_idx=sigma_idx, plot=False,
                                                  plot_points=N, 
                                                  plot_samples_per_point=50, 
                                                  n_rows=len(all_N),
                                                  plot_sigma_idx=np.array([110]),
                                                  plot_counter=1)
    all_errors[i] = relative_error

plt.show()
plt.close()

# Make generalization gap plot
plt.figure(figsize=(6,5))
def fmt(x, pos):
    return f'{x:.0f}'  # Format tick labels

gen_gap_plot(t_steps, 
             all_errors, 
             color_range=all_N//2, 
             cbar_ticks=all_N//2, 
             ylim=[-0.1, 1.0] if not random_data else [-0.05, 0.5], 
             fmt=fmt, 
            )